# First Naive RDD

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from rdrobust import rdrobust

In [ ]:
df_rdd = pd.read_csv('df_rdd_clean.csv')

In [ ]:
df_rdd

# Balance Checks

In [ ]:
def balance_check(df, covariate, running='d_50', cutoff=0, cycle=None):
    """Run rdrobust for one covariate, optionally restricted to one election cycle."""
    data = df.dropna(subset=[covariate, running])
    if cycle is not None:
        data = data[data['election_year'] == cycle]

    result = rdrobust(y=data[covariate], x=data[running], c=cutoff)

    return {
        'covariate': covariate,
        'cycle': cycle if cycle is not None else 'pooled',
        'coef': result.coef.loc['Robust'].values[0],
        'se': result.se.loc['Robust'].values[0],
        'pval': result.pv.loc['Robust'].values[0],
        'ci_lower': result.ci.loc['Robust'].values[0],
        'ci_upper': result.ci.loc['Robust'].values[1],
        'N': int(result.N.sum()),
    }

In [ ]:
# 1. Log population
df_rdd['log_pop'] = np.log(df_rdd['pop_pre'])
results = [balance_check(df_rdd, 'log_pop', cycle=y) for y in [2008, 2014, 2020]]

In [ ]:
# 2. Pre-treatment skill share (level)
results += [balance_check(df_rdd, 'skill_share_pre', cycle=y) for y in [2008, 2014, 2020]]

In [ ]:
# 4. Gender gap in tertiary education
results += [balance_check(df_rdd, 'gender_gap_tertiary', cycle=y) for y in [2008, 2014, 2020]]

In [ ]:
# 5. No-diploma share
results += [balance_check(df_rdd, 'no_diploma_share', cycle=y) for y in [2008, 2014, 2020]]

In [ ]:
# 6. Income median
results += [balance_check(df_rdd, 'income_median', cycle=y) for y in [2008, 2014, 2020]]

In [ ]:
# 7. Income Gini
results += [balance_check(df_rdd, 'income_gini', cycle=y) for y in [2008, 2014, 2020]]

In [ ]:
# 8. Unemployment rate
results += [balance_check(df_rdd, 'unemployment_rate', cycle=y) for y in [2008, 2014, 2020]]

In [ ]:
# ── Assemble final balance table ──
balance_table = pd.DataFrame(results)
balance_table

In [ ]:
def plot_balance(ax, df, covariate, running='d_50', cutoff=0, bins=20):
    data = df.dropna(subset=[covariate, running])
    data = data.assign(bin=pd.cut(data[running], bins=bins))
    binned = data.groupby('bin', observed=True).agg({running: 'mean', covariate: 'mean'}).dropna()

    ax.scatter(binned[running], binned[covariate], s=15, color='steelblue')
    for side, cond in [('left', data[running] < cutoff), ('right', data[running] >= cutoff)]:
        sub = data[cond]
        z = np.polyfit(sub[running], sub[covariate], 1)
        xs = np.linspace(sub[running].min(), sub[running].max(), 50)
        ax.plot(xs, np.polyval(z, xs), color='firebrick', linewidth=1.5)

    ax.axvline(cutoff, color='black', linestyle='--', linewidth=0.8)
    ax.set_title(covariate, fontsize=10)

covariates = ['log_pop', 'skill_share_pre', 'skill_share_trend', 'gender_gap_tertiary',
              'no_diploma_share', 'income_median', 'income_gini', 'unemployment_rate']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, cov in zip(axes.flat, covariates):
    plot_balance(ax, df_rdd, cov)

plt.tight_layout()
plt.show()